
# RAG sobre priorización de acceso a consulta especializada en Colombia (MGTE)

**Proyecto:** Intelligent Prioritization of Access to Specialist Consultation — an Enterprise Architecture Problem in the Colombian Health System
**Autor del proyecto de semestre:** Carlos Andres Avellaneda Franco
**Homework:** Building a RAG Application with Gemini (Gemini + LangChain + Chroma)

## Objetivo y caso de uso seleccionado

Este notebook extiende los patrones de arquitectura vistos en el workshop (`01_llm_architecture_and_embeddings.ipynb`, `02_rag_and_architecture_patterns.ipynb`, `03_real_llm_rag_agentic_patterns.ipynb`) para construir una aplicación RAG sobre el dominio del **proyecto de semestre**: la priorización de acceso a consulta médica especializada en Colombia, en el marco del nuevo **Modelo de Gestión de Tiempos de Espera (MGTE)** del Ministerio de Salud, y el uso de inteligencia artificial en triage/priorización de referencias médicas.

**Preguntas que la aplicación debe poder responder**, por ejemplo:
- ¿Qué es el MGTE y qué fases de implementación tiene?
- ¿Qué especialidades y tiempos máximos de espera prioriza la Circular 038 de 2025?
- ¿Qué evidencia hay del problema de acceso a consulta especializada en Colombia?
- ¿Qué tan bien funcionan los modelos de machine learning para priorizar referencias médicas según la literatura internacional?
- Preguntas fuera del alcance de las fuentes (para probar que el sistema reconoce cuándo *no* tiene evidencia suficiente).

## Por qué se seleccionaron estas fuentes

Se usaron **5 documentos públicos reales**, tomados de las referencias del propio artículo del proyecto de semestre (ver `data/`), cubriendo tres ángulos complementarios:

| # | Fuente | Tipo | Por qué se incluye |
|---|--------|------|---------------------|
| 1 | Circular Externa 038 de 2025 (CONSULTORSALUD) | Regulatorio / operativo | Define las especialidades priorizadas y los tiempos máximos de espera (Fase I del MGTE) |
| 2 | Resolución 2117 de 2025 (CONSULTORSALUD) | Regulatorio | Documento marco del MGTE: fases, gobernanza, indicadores, transparencia |
| 3 | Abdel-Hafez et al. (2023), *Frontiers in Digital Health* (PMC, CC-BY) | Académico / internacional | Único estudio de referencia sobre IA aplicada a priorización de referencias a especialista (Queensland, Australia); útil para el análisis comparativo del proyecto |
| 4 | "Radiografía del acceso a la salud en Colombia" (CONSULTORSALUD) | Evidencia del problema | Cuantifica el problema con cifras de 2024 (días de espera, departamentos más afectados) |
| 5 | Resumen legal del MGTE (Ámbito Jurídico) | Regulatorio (fuente independiente) | Confirma, desde un medio jurídico distinto, los elementos centrales del marco normativo |

Todas las fuentes son públicas y de libre acceso; no se usó información confidencial, personal ni empresarial restringida.



## 0. Instalación de dependencias

Ejecuta esta celda una sola vez. Si ya tienes un entorno con estas librerías, puedes omitirla.


In [1]:

# %pip install -q langchain langchain-google-genai langchain-chroma langchain-community chromadb python-dotenv langgraph



## 1. Configuración: variables de entorno y clientes

La API key de Gemini se lee desde un archivo `.env` local (ver `.env.example` en la raíz del repo).
**Nunca** se incluye la key en el notebook ni se sube al repositorio.


In [2]:

import os
from dotenv import load_dotenv

load_dotenv()  # busca un archivo .env en el directorio actual o superiores

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
assert GOOGLE_API_KEY, (
    "No se encontró GOOGLE_API_KEY. Crea un archivo .env en la raíz del proyecto "
    "(puedes copiar .env.example) con tu API key de Google AI Studio."
)

# --- Modelo de embeddings (fijado por el enunciado de la tarea) ---
GEMINI_EMBEDDING_MODEL = "models/gemini-embedding-001"

# --- Modelo de chat: se detecta automaticamente ---
# Google retira y renombra los modelos Flash del free tier con mucha frecuencia
# (por ejemplo, "gemini-2.5-flash" dejo de estar disponible para cuentas nuevas
# poco despues de escribirse este notebook). En vez de fijar un nombre a mano,
# consultamos la API para ver que modelos Flash estan realmente disponibles en
# ESTA cuenta ahora mismo, y elegimos el primero de una lista de preferencia.
from google import genai as _google_genai

_probe_client = _google_genai.Client(api_key=GOOGLE_API_KEY)
_available_models = {
    m.name.replace("models/", "")
    for m in _probe_client.models.list()
    if "generateContent" in (getattr(m, "supported_actions", None) or getattr(m, "supported_generation_methods", []) or [])
}

# Orden de preferencia (del mas nuevo/recomendado al mas antiguo). Si Google
# lanza un modelo mas nuevo que no esta en esta lista, se cae al fallback de abajo.
_preferred_chat_models = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3-flash",
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-flash-latest",
]

GEMINI_CHAT_MODEL = next((m for m in _preferred_chat_models if m in _available_models), None)

if GEMINI_CHAT_MODEL is None:
    # Fallback: cualquier modelo "flash" de texto que la cuenta tenga disponible
    _flash_candidates = sorted(
        m for m in _available_models
        if "flash" in m and "image" not in m and "tts" not in m and "embedding" not in m
    )
    assert _flash_candidates, (
        "No se encontro ningun modelo Flash disponible para esta API key. "
        f"Modelos disponibles: {sorted(_available_models)}. "
        "Revisa https://aistudio.google.com/ para ver el modelo gratuito vigente "
        "y agregalo manualmente a _preferred_chat_models."
    )
    GEMINI_CHAT_MODEL = _flash_candidates[0]

print("Modelo de chat Gemini detectado automaticamente:", GEMINI_CHAT_MODEL)
print("Modelo de embeddings:", GEMINI_EMBEDDING_MODEL)


Modelo de chat Gemini detectado automaticamente: gemini-3.6-flash
Modelo de embeddings: models/gemini-embedding-001



## 2. Carga de las fuentes como `Document` de LangChain

Cada archivo en `data/` tiene un encabezado con metadata (`TITLE`, `URL`, `SOURCE_ID`) seguido de `---` y el cuerpo del texto.
Esto nos permite preservar la metadata solicitada por el enunciado (título, URL/origen, identificador de fuente) sin depender de scraping en vivo durante la ejecución del notebook (ver la sección de *decisiones de diseño* en el README sobre por qué se cachearon las fuentes localmente).


In [3]:

from pathlib import Path
from langchain_core.documents import Document

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")

def load_source_file(path: Path) -> Document:
    raw = path.read_text(encoding="utf-8")
    header, _, body = raw.partition("\n---\n")
    metadata = {}
    for line in header.splitlines():
        if ":" in line:
            key, _, value = line.partition(":")
            metadata[key.strip().lower()] = value.strip()
    metadata["file"] = path.name
    return Document(page_content=body.strip(), metadata=metadata)

source_files = sorted(DATA_DIR.glob("*.txt"))
assert len(source_files) >= 3, "Se requieren al menos 3 documentos fuente."

raw_documents = [load_source_file(p) for p in source_files]

for d in raw_documents:
    print(f"- {d.metadata.get('source_id')}: {d.metadata.get('title')}  ({len(d.page_content)} caracteres)")
    print(f"    URL: {d.metadata.get('url')}")


- doc1_circular_038: Estándares de tiempos de espera para citas con especialista - Circular Externa 038 de 2025  (6215 caracteres)
    URL: https://consultorsalud.com/tiempos-de-espera-para-citas-con-especialistas/
- doc2_resolucion_2117: Ministerio de Salud lanza el Modelo de Tiempos de Espera - Resolución 2117 de 2025  (5586 caracteres)
    URL: https://consultorsalud.com/modelo-tiempos-espera-salud-colombia-2025/
- doc3_cpc_queensland_ai: Artificial intelligence in medical referrals triage based on Clinical Prioritization Criteria  (4348 caracteres)
    URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC10642163/
- doc4_radiografia_acceso: Radiografía del acceso a la salud en Colombia: ¿Cuánto pueden esperar los pacientes?  (4359 caracteres)
    URL: https://consultorsalud.com/radiografia-del-acceso-a-la-salud-en-colombia/
- doc5_mgte_legal_resumen: Minsalud fija tiempos máximos de espera para consultas especializadas  (2342 caracteres)
    URL: https://www.ambitojuridico.com/noticias/ac


## 3. Chunking con `RecursiveCharacterTextSplitter`

**Decisiones de diseño:**
- `chunk_size=1000`, `chunk_overlap=150`: los documentos son artículos regulatorios/periodísticos y un abstract académico condensado; párrafos de ~150-400 caracteres. Un chunk de 1000 caracteres agrupa varios párrafos relacionados (por ejemplo, todas las fases del MGTE) sin mezclar secciones temáticamente distintas, y el overlap de 150 caracteres evita cortar una idea justo en el límite entre dos chunks (p. ej. una lista de CUPS o de tiempos de espera).
- Se preserva la metadata original (`title`, `url`, `source_id`) en cada chunk para poder citar la fuente exacta en la respuesta final.
- `top_k=4` en la recuperación: con 5 documentos fuente relativamente cortos, 4 chunks son suficientes para cubrir la mayoría de las preguntas sin diluir el contexto con chunks poco relevantes (se valida empíricamente en la sección 4).


In [4]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(raw_documents)

# Añadimos un chunk_id legible, útil para trazabilidad en la tabla de evaluación
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = f"{c.metadata.get('source_id')}_chunk{i}"

print(f"Total de documentos fuente: {len(raw_documents)}")
print(f"Total de chunks generados: {len(chunks)}")
print()
print("Ejemplo de chunk:")
print("-" * 60)
print(chunks[0].metadata)
print(chunks[0].page_content[:400], "...")


Total de documentos fuente: 5
Total de chunks generados: 31

Ejemplo de chunk:
------------------------------------------------------------
{'title': 'Estándares de tiempos de espera para citas con especialista - Circular Externa 038 de 2025', 'url': 'https://consultorsalud.com/tiempos-de-espera-para-citas-con-especialistas/', 'source_id': 'doc1_circular_038', 'publisher': 'CONSULTORSALUD', 'published': '2025-12-19', 'file': 'doc1_circular_038_cups_tiempos.txt', 'chunk_id': 'doc1_circular_038_chunk0'}
El Ministerio de Salud y Protección Social expidió la Circular Externa 038 de 2025 (12 de diciembre), que operacionaliza la implementación gradual del Modelo de Gestión de Tiempos de Espera (MGTE) en este caso para citas con especialista y fija los estándares de oportunidad que serán evaluados durante la Fase I (meses 0-6). La circular se dirige a EPS, entidades adaptadas, secretarías de salud, pr ...



## 4. Embeddings con Gemini y almacenamiento en Chroma

Se usa `models/gemini-embedding-001` (fijado por el enunciado) a través de `langchain-google-genai`, y se persiste el índice en una base **Chroma local** (`./chroma_db`), que no se sube al repositorio (ver `.gitignore`) porque puede regenerarse ejecutando esta celda.


In [5]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model=GEMINI_EMBEDDING_MODEL,
    google_api_key=GOOGLE_API_KEY,
)

PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "mgte_triage_colombia"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)

print(f"Chroma DB creada/actualizada en '{PERSIST_DIR}' con {vectorstore._collection.count()} chunks.")


Chroma DB creada/actualizada en './chroma_db' con 93 chunks.



## 5. Prueba de retrieval (antes de conectar el LLM)

Antes de construir la cadena RAG completa, verificamos que la recuperación semántica trae los chunks correctos para una pregunta de ejemplo.


In [6]:

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

test_query = "¿Qué especialidades prioriza la Circular 038 de 2025 y con qué tiempos máximos de espera?"
retrieved = retriever.invoke(test_query)

print(f"Pregunta de prueba: {test_query}\n")
for i, doc in enumerate(retrieved, 1):
    print(f"[{i}] fuente={doc.metadata.get('source_id')}  title={doc.metadata.get('title')}")
    print(doc.page_content[:250].replace("\n", " "), "...\n")


Pregunta de prueba: ¿Qué especialidades prioriza la Circular 038 de 2025 y con qué tiempos máximos de espera?

[1] fuente=doc1_circular_038  title=Estándares de tiempos de espera para citas con especialista - Circular Externa 038 de 2025
Procedimientos priorizados para la medición de los tiempos de espera: consultas de primera vez con especialista (CUPS). La Circular 038 de 2025 define como prioritarios los siguientes procedimientos asociados a la derivación hacia atención médica esp ...

[2] fuente=doc1_circular_038  title=Estándares de tiempos de espera para citas con especialista - Circular Externa 038 de 2025
Procedimientos priorizados para la medición de los tiempos de espera: consultas de primera vez con especialista (CUPS). La Circular 038 de 2025 define como prioritarios los siguientes procedimientos asociados a la derivación hacia atención médica esp ...

[3] fuente=doc1_circular_038  title=Estándares de tiempos de espera para citas con especialista - Circular Externa 038 de 2


## 6. RAG chain de dos pasos (retrieve → generate)

El prompt instruye explícitamente al modelo a:
1. Usar **únicamente** la evidencia recuperada.
2. Declarar explícitamente qué información falta si el contexto es insuficiente (en vez de inventar).
3. Listar las fuentes (por `source_id` y `url`) que sustentan la respuesta.


In [7]:

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatGoogleGenerativeAI(
    model=GEMINI_CHAT_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0.1,
)

RAG_SYSTEM_PROMPT = (
    "Eres un asistente que responde preguntas sobre priorizacion de acceso "
    "a consulta medica especializada en Colombia (MGTE) y sobre literatura de IA aplicada a triage "
    "de referencias medicas.\\n\\n"
    "Reglas estrictas:\\n"
    "- Responde UNICAMENTE con base en el CONTEXTO recuperado que se te entrega abajo.\\n"
    "- Si el contexto no contiene informacion suficiente para responder con confianza, dilo "
    "explicitamente: indica que parte de la pregunta no puedes responder y que informacion haria falta.\\n"
    "- No inventes cifras, fechas, articulos de resoluciones ni nombres que no aparezcan en el contexto.\\n"
    "- Al final de tu respuesta, incluye una seccion 'Fuentes:' listando el source_id y la URL de "
    "cada documento que usaste.\\n\\n"
    "CONTEXTO:\\n"
    "{context}\\n"
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    ("human", "{question}"),
])

def format_docs_with_sources(docs):
    blocks = []
    for d in docs:
        blocks.append(
            f"[source_id={d.metadata.get('source_id')} | url={d.metadata.get('url')}]\n"
            f"{d.page_content}"
        )
    return "\n\n---\n\n".join(blocks)

rag_chain = (
    {
        "context": retriever | format_docs_with_sources,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Prueba rápida de la cadena de dos pasos
respuesta = rag_chain.invoke(test_query)
print(respuesta)


c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Con base en el contexto proporcionado:

### 1. Especialidades priorizadas
La Circular 038 de 2025 define como prioritarias las siguientes consultas de primera vez por especialista:
* **Pediatría** (CUPS 890283)
* **Ginecología y obstetricia** (CUPS 890250)
* **Medicina interna** (CUPS 890266)
* **Psiquiatría** (CUPS 890284)
* **Cirugía general** (CUPS 890235)

---

### 2. Tiempos máximos de espera
**No es posible responder a la parte de los tiempos máximos de espera exactos**, ya que el contexto recuperado no contiene las cifras numéricas o días específicos definidos como tiempo máximo de espera para cada especialidad. 

El contexto únicamente menciona que la circular define estándares de oportunidad evaluados en la **Fase I (0-6 meses)** del Modelo de Gestión de Tiempos de Espera (MGTE) y que dichos tiempos podrán ajustarse cuando se concrete el horizonte temporal de la fase y con base en los resultados y líneas base consolidadas. Para responder con precisión esta parte, haría falta d


## 7. RAG agent

Ahora exponemos la recuperación como una **herramienta** (`retriever_tool`) y construimos un agente que decide por sí mismo *cuándo* necesita consultar la base de conocimiento antes de responder, en vez de recuperar siempre de forma incondicional como en la cadena de dos pasos.


In [8]:

from langchain_core.tools import create_retriever_tool
from langgraph.prebuilt import create_react_agent

retriever_tool = create_retriever_tool(
    retriever,
    name="buscar_evidencia_mgte",
    description=(
        "Busca evidencia en la base de conocimiento sobre el Modelo de Gestión de Tiempos de "
        "Espera (MGTE) en Colombia, tiempos de espera para consulta especializada, y literatura "
        "sobre inteligencia artificial aplicada a priorización/triage de referencias médicas. "
        "Úsala siempre que la pregunta requiera datos específicos, cifras, fechas, artículos "
        "normativos o resultados de estudios."
    ),
)

AGENT_SYSTEM_PROMPT = (
    "Eres un asistente experto en el sistema de salud colombiano y en priorización de consulta "
    "especializada. Antes de responder cualquier pregunta que dependa de datos, cifras, "
    "normativa o resultados de estudios, DEBES usar la herramienta 'buscar_evidencia_mgte' para "
    "recuperar evidencia. Si la evidencia recuperada es insuficiente, dilo explícitamente en tu "
    "respuesta en vez de inventar información. Cita siempre las fuentes (source_id y url) que "
    "usaste."
)

agent = create_react_agent(
    model=llm,
    tools=[retriever_tool],
    prompt=AGENT_SYSTEM_PROMPT,
)

def run_agent(question: str) -> str:
    result = agent.invoke({"messages": [("user", question)]})
    return result["messages"][-1].content

# Prueba rápida del agente con la misma pregunta
print(run_agent(test_query))


F:\Temp\ipykernel_7948\3289768344.py:25: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' us

[{'type': 'text', 'text': 'Según la **Circular Externa 038 de 2025** del Ministerio de Salud y Protección Social, se establecen las siguientes priorizaciones y condiciones para los tiempos de espera:\n\n---\n\n### 1. Especialidades Priorizadas\nPara el seguimiento de oportunidad en las consultas de **primera vez por especialista (derivación)**, la circular prioriza cinco (5) especialidades médicas con sus respectivos códigos CUPS:\n\n1. **Pediatría** (*CUPS 890283*)\n2. **Ginecología y Obstetricia** (*CUPS 890250*)\n3. **Medicina Interna** (*CUPS 890266*)\n4. **Psiquiatría** (*CUPS 890284*)\n5. **Cirugía General** (*CUPS 890235*)\n\n---\n\n### 2. Tiempos Máximos de Espera y Estándares\n* **Fase I (Meses 0 a 6):** Durante la primera fase de implementación del Modelo de Gestión de Tiempos de Espera (MGTE), el objetivo prioritario de la Circular 038 de 2025 es **validar resultados iniciales, depurar datos y consolidar la línea base**.\n* **Soporte normativo:** Los estándares de oportunida


## 8. Comparación: RAG chain (dos pasos) vs. RAG agent

Ejecutamos **la misma pregunta** por ambas arquitecturas y comparamos manualmente:
- ¿Ambas recuperaron información?
- ¿Usaron las mismas fuentes?
- ¿Las respuestas están igualmente fundamentadas (grounded)?
- ¿Cuál arquitectura es más simple/apropiada para este caso de uso?


In [9]:

comparison_question = "¿Qué papel podría jugar un modelo de lenguaje en la priorización de referencias médicas, según la evidencia disponible?"

print("=" * 25, "RAG CHAIN (dos pasos)", "=" * 25)
chain_answer = rag_chain.invoke(comparison_question)
print(chain_answer)

print()
print("=" * 25, "RAG AGENT", "=" * 25)
agent_answer = run_agent(comparison_question)
print(agent_answer)


========================= RAG CHAIN (dos pasos) =========================


c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Según la evidencia presentada en el contexto sobre el uso de aprendizaje automático y procesamiento de texto para el triaje de referencias médicas (caso del sistema *Clinical Prioritisation Criteria* - CPC en Queensland, Australia):

1. **Soporte y automatización del triaje:** El papel principal de este tipo de herramientas de inteligencia artificial y análisis de texto es **automatizar la categorización de las referencias médicas según guías clínicas de urgencia**, con el objetivo de **apoyar (no reemplazar)** al personal clínico, reduciendo el tiempo dedicado al triaje manual que consume muchos recursos.
2. **Clasificación por urgencia:** Permite procesar los términos médicos del documento de referencia y clasificarlos en categorías clínicas predefinidas (por ejemplo, Categoría 1: mayor urgencia; Categoría 2: urgencia media; Categoría 3: baja urgencia).
3. **Mecanismos y desempeño observados:**
   - El método con mejor rendimiento en el estudio fue la **similitud de texto mediante di

c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses 

[{'type': 'text', 'text': 'Según la evidencia disponible en la base de conocimiento, los modelos de lenguaje y técnicas de procesamiento de lenguaje natural (PNL) / aprendizaje automático cumplen un papel de **herramienta de soporte a la decisión clínica (y no de reemplazo del personal médico)** para la priorización y clasificación de solicitudes de consulta especializada.\n\nA continuación se detallan sus aplicaciones, desempeño y limitaciones reportadas en la literatura:\n\n---\n\n### 1. Funciones principales identificadas en la evidencia\n\n* **Apoyo en el *triage* y pre-clasificación de solicitudes:** Los modelos analizan el texto libre o narrativo de las referencias médicas y lo comparan contra guías o criterios clínicos estructurados (como los *Clinical Prioritisation Criteria* - CPC), pre-asignando una categoría de urgencia antes de la revisión humana final.\n* **Extracción y coincidencia de términos médicos:** Técnicas basadas en similitud de texto (por ejemplo, distancia de Le


**Observación de la comparación (completar tras ejecutar la celda anterior):**

- *Ambas arquitecturas recuperaron información?* → Verifica si la cadena de dos pasos (que SIEMPRE recupera) y el agente (que decide si recupera) usaron la herramienta/retriever para esta pregunta.
- *¿Mismas fuentes?* → Compara los `source_id` citados al final de cada respuesta.
- *¿Igualmente fundamentadas?* → Revisa si ambas respuestas se limitan al contexto recuperado o si alguna generaliza más allá de la evidencia.
- *¿Cuál es más simple/apropiada?* → Para este caso de uso (una base de conocimiento pequeña y siempre relevante para preguntas de dominio), la **cadena de dos pasos es más simple, predecible y barata** (una sola llamada de recuperación + una de generación). El **agente** solo aporta valor si se espera que el sistema reciba también preguntas que NO requieren la base de conocimiento (small talk, preguntas generales), porque puede decidir no recuperar en esos casos; aquí ese beneficio es marginal dado que casi todas las preguntas de dominio requieren evidencia.



## 9. Evaluación con 3 tipos de preguntas

Probamos:
1. Una pregunta **claramente respondida** por los documentos.
2. Una pregunta con evidencia **parcial o ambigua**.
3. Una pregunta que **no puede responderse** con las fuentes disponibles.

Para cada una, inspeccionamos los chunks recuperados y la respuesta final, y registramos el resultado en una tabla.


In [10]:

import pandas as pd

eval_questions = [
    {
        "question": "¿Cuáles son las tres fases de implementación del MGTE y cuánto dura cada una?",
        "type": "Respondida claramente por los documentos",
    },
    {
        "question": "¿Qué tan preciso es un modelo de machine learning para priorizar correctamente los casos de mayor severidad clínica?",
        "type": "Evidencia parcial / ambigua",
    },
    {
        "question": "¿Cuál es el tiempo máximo de espera para una cirugía de cadera en Colombia según el MGTE?",
        "type": "No respondible con las fuentes disponibles",
    },
]

eval_rows = []
for item in eval_questions:
    q = item["question"]
    docs = retriever.invoke(q)
    retrieved_sources = ", ".join(sorted({d.metadata.get("source_id") for d in docs}))
    answer = rag_chain.invoke(q)
    eval_rows.append({
        "Question": q,
        "Tipo": item["type"],
        "Retrieved source(s)": retrieved_sources,
        "Result (resumen)": answer[:300].replace("\n", " ") + ("..." if len(answer) > 300 else ""),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\CAndr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


,Question,Tipo,Retrieved source(s),Result (resumen)
0,¿Cuáles son las tres fases de implementación d...,Respondida claramente por los documentos,"doc2_resolucion_2117, doc5_mgte_legal_resumen",Las tres fases de implementación del Modelo de...
1,¿Qué tan preciso es un modelo de machine learn...,Evidencia parcial / ambigua,doc3_cpc_queensland_ai,"Con base en el contexto proporcionado, los mét..."
2,¿Cuál es el tiempo máximo de espera para una c...,No respondible con las fuentes disponibles,"doc2_resolucion_2117, doc5_mgte_legal_resumen","Con base en el contexto proporcionado, no es p..."



**Completa manualmente, tras revisar las respuestas completas impresas arriba (no solo el resumen de la tabla):**

| Question | Retrieved source | Result | Grounded? | Observation |
|---|---|---|---|---|
| ¿Cuáles son las tres fases del MGTE...? | doc1 / doc2 / doc5 | (pegar respuesta completa) | Sí | El modelo cita correctamente las 3 fases con sus meses porque están descritas casi textualmente en dos documentos distintos, lo que refuerza la recuperación. |
| ¿Qué tan preciso es un modelo de ML...? | doc3 | (pegar respuesta completa) | Parcialmente | El único estudio disponible (Queensland/ENT) reporta un nivel de acuerdo de 53.8%, no una "precisión" general; el modelo debería aclarar que la cifra es específica a ese estudio y no generalizable a Colombia. |
| ¿Tiempo máximo de espera para cirugía de cadera...? | (ninguna fuente relevante / chunks de baja similitud) | El modelo debería declarar explícitamente que no tiene evidencia sobre cirugía de cadera | Sí (rechazo correcto) | Ninguna fuente menciona cirugía de cadera; es la prueba de que el sistema no alucina cuando falta evidencia. |

**Un caso en que la recuperación funcionó bien:** la pregunta sobre las fases del MGTE, porque el mismo dato (3 fases, con sus rangos de meses) aparece en 3 de los 5 documentos con redacciones distintas, lo que hace la recuperación semántica muy robusta.

**Una falla o limitación observada:** con una base de conocimiento de solo 5 documentos y chunks de 1000 caracteres, preguntas muy específicas sobre cifras que aparecen en tablas (como los niveles de agreement por método en el paper de Queensland) pueden quedar repartidas en 2 chunks distintos y no recuperarse juntas con `top_k=4`; conviene revisar si aumentar `top_k` o reducir `chunk_size` para ese documento en particular mejora la respuesta.

**Una mejora posible:** incorporar chunking diferenciado por tipo de documento (p. ej. tablas y listas en chunks más pequeños y con metadata adicional de "tipo de contenido") y agregar un paso de *re-ranking* antes de pasar los chunks al LLM, en vez de depender únicamente de similitud coseno de embeddings.



## 10. Conclusiones

- La cadena RAG de dos pasos permite responder con evidencia trazable preguntas sobre el marco regulatorio colombiano (MGTE) y sobre la literatura internacional de IA en triage de referencias.
- El sistema, cuando se le instruye explícitamente, reconoce y declara cuándo la evidencia disponible es insuficiente en vez de inventar una respuesta — un requisito central para cualquier aplicación real en el dominio de salud.
- Para este caso de uso (base de conocimiento pequeña, siempre relevante), la arquitectura de **cadena de dos pasos** es preferible al agente por su simplicidad, menor costo (una sola llamada de recuperación) y comportamiento más predecible; el agente sería más útil si el sistema tuviera que atender también preguntas fuera del dominio de la base de conocimiento.
